In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/544 Project/"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install -r requirements.txt

In [ ]:
import random
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from peft import PeftModel, PeftConfig

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [ ]:
# ADD Hugging Face API KEY instead of HF_TOKEN
login(token="HF_TOKEN")

In [ ]:
# ADD "qwen" or "gemma" instead of "GENERATOR_MODEL_ID" and "CHECKER_MODEL_ID"
GENERATOR = "GENERATOR_MODEL_ID"
CHECKER = "CHECKER_MODEL_ID"

In [ ]:
LORA_PATHS = {
    "qwen": {
        "model": "./train/qwen/model_files/model",
        "tokenizer": "./train/qwen/model_files/tokenizer",
    },
    "gemma": {
        "model": "./train/gemma/model_files/model",
        "tokenizer": "./train/gemma/model_files/tokenizer",
    },
}

In [ ]:
DATA_PATHS = {
    "qa": "./data/test/halueval_qa_data.xlsx",
    "dialogue": "./data/test/halueval_dialogue_data.xlsx",
    "summarization": "./data/test/halueval_summarization_data.xlsx",
}

In [ ]:
INSTRUCTION_PATHS = {
    "qa": "./data/test/instruction_files/qa_evaluation_instruction.txt",
    "dialogue": "./data/test/instruction_files/dialogue_evaluation_instruction.txt",
    "summarization": "./data/test/instruction_files/summarization_evaluation_instruction.txt",
}

In [ ]:
MAX_SAMPLES = None
MAX_REFINEMENT_ROUNDS = 1
MAX_NEW_TOKENS_GEN = 256
MAX_NEW_TOKENS_CHECK = 128
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.9

In [ ]:
TASKS = ["qa", "dialogue", "summarization"]
SUMMARY_PATH = "./test_results/results_generator_checker.xlsx"

In [ ]:
def load_instruction(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

In [ ]:
INSTRUCTIONS = {task: load_instruction(p) for task, p in INSTRUCTION_PATHS.items()}

In [ ]:
GENERATOR_SYSTEM = (
    "You are a helpful, factual assistant. "
    "Only make claims supported by the provided context. "
    "If the context lacks sufficient information, say so instead of guessing."
)

In [ ]:
def build_generator_initial(task: str, row: dict) -> list:
    # Generator can see only raw content
    if task == "qa":
        user = (
            "Answer the following question based only on the provided knowledge.\n\n"
            f"Knowledge: {row.get('knowledge', '')}\n"
            f"Question: {row['question']}\n\nAnswer:"
        )
    elif task == "dialogue":
        user = (
            "Continue the dialogue with a natural, factual response "
            "based only on the provided knowledge.\n\n"
            f"Knowledge: {row.get('knowledge', '')}\n"
            f"Dialogue History:\n{row['dialogue_history']}\n\nResponse:"
        )
    elif task == "summarization":
        user = (
            "Write a concise, faithful summary of the document below. "
            "Do not add information not present in the document.\n\n"
            f"Document:\n{row['document']}\n\nSummary:"
        )
    return [
        {"role": "system", "content": GENERATOR_SYSTEM},
        {"role": "user",   "content": user},
    ]


In [ ]:
def build_generator_refinement(task: str, row: dict, previous_response: str, checker_feedback: str) -> list:
    # Generator sees its previous response and the checkers specific critique and is asked to revise
    if task == "qa":
        context_block = (
            f"Knowledge: {row.get('knowledge', '')}\n"
            f"Question: {row['question']}"
        )
        output_label = "Revised Answer:"
    elif task == "dialogue":
        context_block = (
            f"Knowledge: {row.get('knowledge', '')}\n"
            f"Dialogue History:\n{row['dialogue_history']}"
        )
        output_label = "Revised Response:"
    elif task == "summarization":
        context_block = f"Document:\n{row['document']}"
        output_label  = "Revised Summary:"

    user = (
        f"{context_block}\n\n"
        f"Your previous response was:\n{previous_response}\n\n"
        f"A fact-checker reviewed it and raised the following concerns:\n"
        f"{checker_feedback}\n\n"
        f"Please revise your response to address these concerns. "
        f"Only include claims supported by the context above.\n\n"
        f"{output_label}"
    )
    return [
        {"role": "system", "content": GENERATOR_SYSTEM},
        {"role": "user", "content": user},
    ]

In [ ]:
CHECKER_SUFFIX = (
    "\n\nFirst, state your judgement as exactly 'Yes' (contains hallucinations) "
    "or 'No' (factually consistent).\n"
    "If 'Yes', on the next line write: "
    "'Feedback: <which specific claims are unsupported or incorrect and why>.'\n"
    "If 'No', no feedback is needed.\n"
    "#Your Judgement#:"
)

In [ ]:
def build_checker_messages(task: str, row: dict, response: str, instruction: str) -> list:
    # Checker sees original context and generated response for which it gives a verdict and feedback
    if task == "qa":
        user = (
            instruction
            + "\n\n#Question#: " + row["question"]
            + "\n#Answer#: " + response
            + CHECKER_SUFFIX
        )
    elif task == "dialogue":
        user = (
            instruction
            + "\n\n#Dialogue History#: " + row["dialogue_history"]
            + "\n#Response#: " + response
            + CHECKER_SUFFIX
        )
    elif task == "summarization":
        user = (
            instruction
            + "\n\n#Document#: " + row["document"]
            + "\n#Summary#: " + response
            + CHECKER_SUFFIX
        )
    return [{"role": "user", "content": user}]

In [ ]:
def parse_checker_output(raw: str) -> tuple:
  # Checker gives a judgement Yes, No or failed and a feedback
    lines = raw.strip().splitlines()
    verdict  = "failed"
    feedback = ""

    for i, line in enumerate(lines):
        stripped = line.strip().replace(".", "")
        has_yes  = "Yes" in stripped
        has_no = "No"  in stripped

        if has_yes and not has_no:
            verdict = "Yes"
            rest = "\n".join(lines[i + 1:]).strip()
            if "Feedback:" in rest:
                feedback = rest.split("Feedback:", 1)[1].strip()
            elif rest:
                feedback = rest
            break
        elif has_no and not has_yes:
            verdict  = "No"
            feedback = ""
            break

    return verdict, feedback

In [ ]:
def load_lora_model(name: str):
  # Initalizing the lora model
    cfg = LORA_PATHS[name]
    lora_path = cfg["model"]
    tok_path  = cfg["tokenizer"]

    print(f"\n{'='*60}")
    print(f"Loading '{name}' LoRA model")
    print(f"Adapter : {lora_path}")

    peft_cfg = PeftConfig.from_pretrained(lora_path)
    base_model_id = peft_cfg.base_model_name_or_path
    print(f"  Base : {base_model_id}")

    base = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, lora_path)
    model = model.merge_and_unload()
    model.eval()
    total_params = 0
    for p in model.parameters():
        num_params = p.numel()
        total_params += num_params
    print(f"Params : {total_params:,}")

    tokenizer = AutoTokenizer.from_pretrained(tok_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    print(f"Tokenizer ready")

    return model, tokenizer, base_model_id

In [ ]:
def generate_single(messages: list, model, tokenizer, max_new_tokens: int,
                    temperature: float = 0.7, top_p: float = 0.9, greedy: bool = False) -> str:
    has_template = (
        hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
    )
    if has_template:
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
    else:
        parts = []
        allowed_roles = ("system", "user")

        for m in messages:
            if m["role"] in allowed_roles:
                parts.append(m["content"])
        input_text = "\n\n".join(parts)

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )


    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=not greedy,
            temperature=temperature if not greedy else 1.0,
            top_p=top_p if not greedy else 1.0,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.decode(output_ids[0, input_len:], skip_special_tokens=True)

    del inputs, output_ids
    torch.cuda.empty_cache()
    return decoded.strip()

In [ ]:
def run_refinement_loop(task: str, row: dict, gen_model, gen_tok, chk_model, chk_tok, instruction: str, initial_response: str) -> dict:
    # Check for the judgement from last run and refine the output if it was hallucinated
    history = []
    current_response = initial_response
    last_feedback = ""
    final_judgement = "failed"

    for round_idx in range(1, MAX_REFINEMENT_ROUNDS + 2):
        # Checker
        try:
            chk_msgs = build_checker_messages(task, row, current_response, instruction)
            checker_raw = generate_single(
                chk_msgs, chk_model, chk_tok,
                max_new_tokens=MAX_NEW_TOKENS_CHECK,
                greedy=True,
            )
            judgement, feedback = parse_checker_output(checker_raw)
        except Exception as e:
            print(f"\n [Round {round_idx}] CHECKER ERROR: {e}")
            checker_raw = ""
            judgement = "failed"
            feedback = ""

        history.append({
            "round": round_idx,
            "response": current_response,
            "checker_raw": checker_raw,
            "judgement": judgement,
            "feedback": feedback,
        })
        final_judgement = judgement

        # Exit early if checker is satisfied or no feedback to act on
        if judgement == "No" or judgement == "failed" or not feedback:
            break

        if round_idx > MAX_REFINEMENT_ROUNDS:
            break

        # Generator
        try:
            gen_msgs = build_generator_refinement(
                task, row, current_response, feedback
            )
            current_response = generate_single(
                gen_msgs, gen_model, gen_tok,
                max_new_tokens=MAX_NEW_TOKENS_GEN,
                temperature=GEN_TEMPERATURE,
                top_p=GEN_TOP_P,
            )
        except Exception as e:
            print(f"\n [Round {round_idx}] GENERATOR ERROR: {e}")
            break

    return {
        "final_response": current_response,
        "final_judgement": final_judgement,
        "rounds_used": len(history),
        "history": history,
    }

In [ ]:
def build_eval_rows(df: pd.DataFrame, task: str, rng: random.Random) -> list:
    eval_rows = []
    for _, row in df.iterrows():
        use_hallucinated = rng.random() > 0.5
        if task == "qa":
            base = {"knowledge": row["knowledge"], "question": row["question"]}
            base["answer"] = row["hallucinated_answer"] if use_hallucinated else row["right_answer"]
            base["ground_truth"] = "Yes" if use_hallucinated else "No"
        elif task == "dialogue":
            base = {"knowledge": row["knowledge"], "dialogue_history": row["dialogue_history"]}
            base["response"] = row["hallucinated_response"] if use_hallucinated else row["right_response"]
            base["ground_truth"] = "Yes" if use_hallucinated else "No"
        elif task == "summarization":
            base = {"document": row["document"]}
            base["summary"] = row["hallucinated_summary"] if use_hallucinated else row["right_summary"]
            base["ground_truth"] = "Yes" if use_hallucinated else "No"
        eval_rows.append(base)
    return eval_rows

In [ ]:
def run_task(task: str, gen_model, gen_tok, gen_base_id: str, chk_model, chk_tok, chk_base_id: str) -> dict:
    print(f"Task={task} | Generator={GENERATOR} | Checker={CHECKER}")
    df = pd.read_excel(DATA_PATHS[task])
    print(f"  Loaded {len(df)} rows")
    if MAX_SAMPLES and MAX_SAMPLES < len(df):
        df = df.sample(MAX_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)
        print(f"  Subsampled to {len(df)} rows")
    rng = random.Random(RANDOM_SEED)
    eval_rows = build_eval_rows(df, task, rng)
    instruction = INSTRUCTIONS[task]
    yes_count = sum(1 for r in eval_rows if r["ground_truth"] == "Yes")
    print(f"Hallucinated={yes_count} Correct={len(eval_rows) - yes_count}")

    out_path = Path(f"./test_results/generator_checker/{GENERATOR}_gen_{CHECKER}_chk_{task}.jsonl")
    out_path.parent.mkdir(parents=True, exist_ok=True)

    correct = 0
    incorrect = 0
    failed = 0
    detected = 0
    corrected = 0
    results = []

    with out_path.open("w", encoding="utf-8") as fout:
        for row in tqdm(eval_rows, desc=f"{task}"):
            ground_truth = row["ground_truth"]

            if task == "qa":
                initial_response = row["answer"]
            elif task == "dialogue":
                initial_response = row["response"]
            elif task == "summarization":
                initial_response = row["summary"]

            loop_out = run_refinement_loop(
                task, row,
                gen_model, gen_tok,
                chk_model, chk_tok,
                instruction,
                initial_response,
            )
            judgement = loop_out["final_judgement"]

            if judgement == "failed":
                failed += 1
                incorrect += 1
            elif judgement == ground_truth:
                correct += 1
            else:
                incorrect += 1

            # Detection and Correcting
            history = loop_out["history"]
            if ground_truth == "Yes" and len(history) >= 1:
                round1_judgement = history[0]["judgement"]
                if round1_judgement == "Yes":
                    detected += 1
                    if judgement == "No" and len(history) >= 2:
                        corrected += 1

            record = {
                **row,
                "initial_response": initial_response,
                "final_response": loop_out["final_response"],
                "final_judgement": judgement,
                "rounds_used": loop_out["rounds_used"],
                "history": loop_out["history"],
                "generator": GENERATOR,
                "checker": CHECKER,
            }
            results.append(record)
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"\nDone. {correct} correct | {incorrect} incorrect "
          f"(incl. {failed} failed) | Total {len(eval_rows)}")

    avg_rounds = sum(r["rounds_used"] for r in results) / len(results) if results else 0
    print(f"Avg rounds per sample : {avg_rounds:.2f}  "
          f"(1 = checker accepted on first try)")

    n_hallucinated = yes_count
    if n_hallucinated != 0:
        detection_rate = detected / n_hallucinated
    else:
        detection_rate = 0.0
    if detected != 0:
        correction_rate = corrected / detected
    else:
        correction_rate = 0.0
    print(f"Detection rate (checker caught hall. in round 1) : {detection_rate:.2%} ({detected}/{n_hallucinated})")
    print(f"Correction rate (generator fixed after feedback) : {correction_rate:.2%} ({corrected}/{detected})")

    valid = []
    y_true = []
    y_pred = []

    for r in results:
        if r["final_judgement"] != "failed":
            valid.append(r)
            y_true.append(r["ground_truth"])
            y_pred.append(r["final_judgement"])
    total = len(results)
    n_valid = len(valid)

    if total != 0:
        acc_all = correct / total
    else:
        acc_all = 0.0
    if n_valid != 0:
        acc_valid = accuracy_score(y_true, y_pred)
    else:
        acc_valid = 0.0
    prec = precision_score(y_true, y_pred, pos_label="Yes", zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label="Yes", zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label="Yes", zero_division=0)

    if n_valid:
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, digits=4))
        labels = ["Yes", "No"]
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        row_labels = [f"True_{l}" for l in labels]
        col_labels = [f"Pred_{l}" for l in labels]

        df = pd.DataFrame(cm, index=row_labels, columns=col_labels)
        print(df)

    return {
        "task": task,
        "generator": GENERATOR,
        "checker": CHECKER,
        # "gen_base_model": gen_base_id,
        # "chk_base_model": chk_base_id,
        "model_source": "gc-lora",
        "max_rounds": MAX_REFINEMENT_ROUNDS,
        # "total": total,
        # "valid": n_valid,
        # "failed": total - n_valid,
        # "correct": correct,
        # "incorrect": incorrect,
        # "avg_rounds": round(avg_rounds, 2),
        # "accuracy_all": round(acc_all, 4),
        # "accuracy_valid": round(acc_valid, 4),
        # "precision_yes": round(prec, 4),
        # "recall_yes": round(rec, 4),
        # "f1_yes": round(f1, 4),
        "detected": detected,
        "corrected": corrected,
        "detection_rate": round(detection_rate, 4),
        "correction_rate": round(correction_rate, 4),
    }

In [ ]:
print(f"\nConfiguration : Generator={GENERATOR} → Checker={CHECKER}")
print(f"Refinement rounds : {MAX_REFINEMENT_ROUNDS}")

In [ ]:
gen_model, gen_tok, gen_base_id = load_lora_model(GENERATOR)

In [ ]:
if CHECKER == GENERATOR:
    print(f"\nGenerator and Checker are the same model — reusing weights.")
    chk_model, chk_tok, chk_base_id = gen_model, gen_tok, gen_base_id
else:
    chk_model, chk_tok, chk_base_id = load_lora_model(CHECKER)

In [ ]:
all_summaries = []
for task in TASKS:
    summary = run_task(
        task,
        gen_model, gen_tok, gen_base_id,
        chk_model, chk_tok, chk_base_id,
    )
    all_summaries.append(summary)

    new_df = pd.DataFrame(all_summaries)
    if Path(SUMMARY_PATH).exists():
        existing = pd.read_excel(SUMMARY_PATH)
        new_df = pd.concat([existing, new_df], ignore_index=True)
        new_df = new_df.drop_duplicates(
            subset=["task", "generator", "checker"], keep="last"
        )
    new_df.to_excel(SUMMARY_PATH, index=False)
    print(f"\n  Summary saved → {SUMMARY_PATH}")


In [ ]:
print(f"DONE  [{GENERATOR} → {CHECKER}]  —  all 3 tasks complete")
pd.DataFrame(all_summaries)